# Entrenamiento de los folds 2, 3 y 4: baseline y v1 (λ = 0.2), 100 épocas

**Objetivo:** completar la validación cruzada de 5 folds (con los folds 0 y 1 ya disponibles) entrenando baseline y v1 en los folds 2, 3 y 4 en el mismo runtime, de forma consecutiva.

**Diseño tolerante a interrupciones de Colab y Drive:**
- La copia del dataset preprocesado se descarga **una sola vez** mediante la API de Drive (sin FUSE). Es el mismo dataset de 1000 casos para los tres folds; solo cambia el split interno.
- Antes de lanzar cada entrenamiento (fold y trainer), el notebook **comprueba en Drive, mediante la API**, si esa combinación está completa (200/200 predicciones de validación) y, en tal caso, la omite. Así, ante una interrupción durante el fold 3, basta con volver a ejecutar el notebook completo: retoma el proceso sin repetir lo ya terminado.
- Se omite `--npz`, que no es necesario para la evaluación y solo consume disco adicional.

Este notebook **no evalúa**: la evaluación se realiza en un notebook aparte (como el 32) una vez disponibles las predicciones.

**Nota de ejecución:** el notebook se ejecuta celda a celda, hasta la sección 13 inclusive, y se detiene para revisar el resumen antes de lanzar la sección 14 (entrenamiento real). La sección 13 es una parada de seguridad deliberada: no entrena, y permite confirmar que dataset, split y trainer v1 coinciden con los empleados en los folds 0 y 1 antes de comprometer varias horas de GPU.


## 0. Comprobación de la GPU


In [1]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Activa una GPU (A100 recomendado) antes de continuar.")
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")


PyTorch: 2.11.0+cu128
CUDA disponible: True
GPU: NVIDIA A100-SXM4-40GB
VRAM: 42.41 GB


## 1. Instalación de dependencias

Si el runtime procede de una sesión anterior con todo instalado, la celda es rápida y no es necesario reiniciar.


In [2]:
!python -m pip install -q nnunetv2 medpy scikit-image SimpleITK
print("Dependencias instaladas.")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.3/156.3 kB 16.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 291.1/291.1 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.8/102.8 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.8/76.8 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.6/53.6 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.1/28.1 MB 99.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 111.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 120.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 1b. Reinicio del runtime (solo tras instalar en un runtime nuevo)

Si el runtime procede de una sesión anterior, esta celda puede omitirse.


In [ ]:
import os
os.kill(os.getpid(), 9)


## 2. Autenticación con la API de Drive (descargas pesadas) y montaje de Drive


In [1]:
from google.colab import auth
auth.authenticate_user()

import google.auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

creds, _ = google.auth.default()
drive_api = build("drive", "v3", credentials=creds)
print("Api de Drive autenticada.")

from google.colab import drive
drive.mount("/content/drive", force_remount=True)
print("Drive montado (solo para split/resultados, no para la copia pesada).")

# El timeout de socket se activa después de autenticar; si se activa antes
# puede interferir con la propagación de credenciales de Colab
import socket
socket.setdefaulttimeout(90)
print("Timeout de socket (90s) activado para las descargas.")


Api de Drive autenticada.
Mounted at /content/drive
Drive montado (solo para split/resultados, no para la copia pesada).
Timeout de socket (90s) activado para las descargas.


## 3. Definición de rutas y configuración (FOLDS = 2, 3, 4)


In [2]:
import os, shutil, json, time, io, subprocess
from pathlib import Path

BASE = Path("/content/drive/MyDrive/TFM_nnUNet")

LOCAL_PREP_ROOT = Path("/content/nnUNet_preprocessed")
LOCAL_DATASET = LOCAL_PREP_ROOT / "Dataset503_ImageCAS1000"

NNUNET_RESULTS = BASE / "nnUNet_results"
CUSTOM_TRAINERS = BASE / "custom_trainers"
CUSTOM_TRAINERS.mkdir(parents=True, exist_ok=True)

DATASET_ID = 503
DATASET_FOLDER_NAME = "Dataset503_ImageCAS1000"
CONFIGURATION = "3d_fullres"
FOLDS = [2, 3, 4]

BASELINE_TRAINER = "nnUNetTrainer_100epochs"
V1_TRAINER = "nnUNetTrainerV1_1000casos_100epochs"
WEIGHT_DICE = 1.0
WEIGHT_CE = 1.0
LAMBDA_CLDICE = 0.2
SKELETON_ITERATIONS = 10

SPLIT_CORRECTO = BASE / "splits_final_Dataset503_1000_auto5fold.json"
assert SPLIT_CORRECTO.exists(), "Falta splits_final_Dataset503_1000_auto5fold.json en BASE."

print("Local dataset:", LOCAL_DATASET)
print("Resultados:", NNUNET_RESULTS)
print("Folds a entrenar:", FOLDS)


Local dataset: /content/nnUNet_preprocessed/Dataset503_ImageCAS1000
Resultados: /content/drive/MyDrive/TFM_nnUNet/nnUNet_results
Folds a entrenar: [2, 3, 4]


## 4. Localización de las carpetas en Drive mediante API y listado de sus archivos


In [3]:
def buscar_id(nombre, parent_id, carpeta=True):
    mime = " and mimeType = 'application/vnd.google-apps.folder'" if carpeta else ""
    q = f"name = '{nombre}' and '{parent_id}' in parents and trashed = false{mime}"
    res = drive_api.files().list(q=q, fields="files(id, name)", pageSize=5).execute()
    archivos = res.get("files", [])
    if not archivos:
        raise FileNotFoundError(f"No se encontro '{nombre}' dentro de {parent_id}")
    return archivos[0]["id"]

def listar_hijos(parent_id):
    archivos = []
    page_token = None
    while True:
        res = drive_api.files().list(
            q=f"'{parent_id}' in parents and trashed = false",
            fields="nextPageToken, files(id, name, size, mimeType)",
            pageSize=1000,
            pageToken=page_token,
        ).execute()
        archivos.extend(res.get("files", []))
        page_token = res.get("nextPageToken")
        if not page_token:
            break
    return archivos

id_tfm = buscar_id("TFM_nnUNet", "root")
id_prep = buscar_id("nnUNet_preprocessed", id_tfm)
id_dataset = buscar_id("Dataset503_ImageCAS1000", id_prep)
id_cfg = buscar_id("nnUNetPlans_3d_fullres", id_dataset)
id_gt = buscar_id("gt_segmentations", id_dataset)

archivos_raiz_dataset = listar_hijos(id_dataset)
archivos_cfg = listar_hijos(id_cfg)
archivos_gt = listar_hijos(id_gt)

archivos_json_top = [f for f in archivos_raiz_dataset if f["name"] in
                      ("nnUNetPlans.json", "dataset.json", "dataset_fingerprint.json")]

print("JSON de nivel superior encontrados:", [f["name"] for f in archivos_json_top])
print("Archivos en nnUNetPlans_3d_fullres:", len(archivos_cfg))
print("Archivos en gt_segmentations:", len(archivos_gt))


JSON de nivel superior encontrados: ['nnUNetPlans.json', 'dataset.json', 'dataset_fingerprint.json']
Archivos en nnUNetPlans_3d_fullres: 3000
Archivos en gt_segmentations: 1000


## 5. Verificación de que Dataset503 en Drive está completo (1000/1000, por metadatos)


In [4]:
imgs = [f for f in archivos_cfg if f["name"].startswith("ImageCAS_") and f["name"].endswith(".b2nd") and not f["name"].endswith("_seg.b2nd")]
segs = [f for f in archivos_cfg if f["name"].endswith("_seg.b2nd")]
pkls = [f for f in archivos_cfg if f["name"].endswith(".pkl")]
gts = [f for f in archivos_gt if f["name"].endswith(".nii.gz")]

print("Imagenes:", len(imgs), "Segmentaciones:", len(segs), "PKL:", len(pkls), "GT:", len(gts))
assert len(imgs) == 1000 and len(segs) == 1000 and len(pkls) == 1000 and len(gts) == 1000
assert len(archivos_json_top) == 3
print("\nDataset503 en Drive: 1000/1000 y JSON de planificacion presentes.")


Imagenes: 1000 Segmentaciones: 1000 PKL: 1000 GT: 1000

Dataset503 en Drive: 1000/1000 y JSON de planificacion presentes.


## 6. Comprobación del espacio local


In [5]:
usage = shutil.disk_usage("/content")
free_gb = usage.free / 1024**3
print(f"Espacio libre local: {free_gb:.1f} GB")
if free_gb < 40:
    raise RuntimeError(f"Solo hay {free_gb:.1f} GB libres. Revisa antes de continuar.")
print("Espacio local suficiente (o copia ya existente).")


Espacio libre local: 187.8 GB
Espacio local suficiente (o copia ya existente).


## 7. Descarga robusta mediante API: versión secuencial (referencia) y paralela

La descarga se realiza **una sola vez** y sirve para entrenar los tres folds, ya que el dataset completo es el mismo (solo cambia el split interno por fold). Si el runtime se reinicia y la celda se vuelve a ejecutar, no se repite lo ya descargado y verificado por tamaño.

La descarga secuencial de este mismo dataset requirió ≈ 9.3 h (334 min para 3000 archivos y 223.8 min para 1000 GT), a un ritmo de 6.7-13.4 s/archivo, dominado por la latencia de cada petición HTTP y no por el ancho de banda. Este tipo de cuello de botella mejora con descargas **en paralelo** (varias peticiones simultáneas).

Se implementan ambas versiones:
- `descargar_lista` (secuencial): se conserva como referencia para el benchmark.
- `descargar_lista_paralelo`: utiliza 8 workers por defecto, cada uno con **su propio cliente de Drive** (para no compartir entre hilos un objeto que no es thread-safe). Descarga a un archivo temporal `.part` y lo renombra al definitivo solo tras verificar el tamaño (evitando archivos a medio escribir ante un fallo), con reintentos y **backoff exponencial** (5 s, 10 s, 20 s... hasta un máximo de 60 s), y omite automáticamente los archivos locales ya completos.

La aceleración no se asume de antemano: se mide con un benchmark real antes de lanzar la descarga completa (celda siguiente).


In [6]:
def descargar_lista(archivos_remotos, carpeta_local, max_reintentos=5):
    """Version secuencial (un archivo tras otro). Se mantiene como referencia
    para el benchmark de la siguiente celda."""
    carpeta_local.mkdir(parents=True, exist_ok=True)
    pendientes = list(archivos_remotos)
    intento = 0
    while pendientes and intento < max_reintentos:
        intento += 1
        fallidos = []
        for i, meta in enumerate(pendientes):
            destino = carpeta_local / meta["name"]
            tam_esperado = int(meta.get("size", 0) or 0)
            try:
                if destino.exists() and (tam_esperado == 0 or destino.stat().st_size == tam_esperado):
                    continue
                request = drive_api.files().get_media(fileId=meta["id"])
                fh = io.FileIO(destino, "wb")
                downloader = MediaIoBaseDownload(fh, request, chunksize=1024 * 1024 * 20)
                done = False
                while not done:
                    status, done = downloader.next_chunk(num_retries=3)
                fh.close()
            except Exception as e:
                fallidos.append(meta)
        pendientes = fallidos
        if pendientes:
            time.sleep(15)
    return pendientes


import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

_thread_local = threading.local()

def _drive_service_del_hilo():
    """Cada hilo construye y reutiliza su propio cliente de Drive -- un
    objeto de la API de Drive compartido entre hilos no es seguro."""
    if not hasattr(_thread_local, "service"):
        _thread_local.service = build("drive", "v3", credentials=creds)
    return _thread_local.service

def _descargar_un_archivo(meta, carpeta_local, max_reintentos=5):
    destino = carpeta_local / meta["name"]
    tam_esperado = int(meta.get("size", 0) or 0)
    if destino.exists() and (tam_esperado == 0 or destino.stat().st_size == tam_esperado):
        return ("skip", meta["name"], None)

    parcial = destino.with_name(destino.name + ".part")
    espera = 5
    for intento in range(1, max_reintentos + 1):
        try:
            service = _drive_service_del_hilo()
            request = service.files().get_media(fileId=meta["id"])
            fh = io.FileIO(parcial, "wb")
            downloader = MediaIoBaseDownload(fh, request, chunksize=1024 * 1024 * 20)
            done = False
            while not done:
                status, done = downloader.next_chunk(num_retries=3)
            fh.close()
            if tam_esperado and parcial.stat().st_size != tam_esperado:
                raise IOError(f"Tamano incorrecto: {parcial.stat().st_size} != {tam_esperado}")
            parcial.replace(destino)
            return ("ok", meta["name"], None)
        except Exception as e:
            if parcial.exists():
                parcial.unlink(missing_ok=True)
            if intento == max_reintentos:
                return ("fail", meta["name"], f"{type(e).__name__}: {e}")
            time.sleep(espera)
            espera = min(espera * 2, 60)

def descargar_lista_paralelo(archivos_remotos, carpeta_local, n_workers=8, max_reintentos=5, silencioso=False):
    carpeta_local.mkdir(parents=True, exist_ok=True)
    if not silencioso:
        print(f"Total: {len(archivos_remotos)} archivos, {n_workers} workers en paralelo")

    ok = skip = 0
    fallidos = []
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=n_workers) as executor:
        futuros = {executor.submit(_descargar_un_archivo, meta, carpeta_local, max_reintentos): meta
                   for meta in archivos_remotos}
        completados = 0
        for futuro in as_completed(futuros):
            estado, nombre, error = futuro.result()
            completados += 1
            if estado == "ok":
                ok += 1
            elif estado == "skip":
                skip += 1
            else:
                fallidos.append((nombre, error))
            if not silencioso and (completados % 100 == 0 or completados == len(archivos_remotos)):
                print(f"  {completados}/{len(archivos_remotos)} (ok={ok} skip={skip} fail={len(fallidos)})")

    dt = time.time() - t0
    if not silencioso:
        print(f"Terminado en {dt/60:.2f} min. ok={ok} skip={skip} fail={len(fallidos)}")
        for nombre, error in fallidos[:10]:
            print(f"  fallo en {nombre}: {error}")
    return fallidos, dt

print("Funciones de descarga (secuencial + paralela) listas.")


Funciones de descarga (secuencial + paralela) listas.


## 7b. Benchmark: descarga secuencial frente a paralela (100 + 100 archivos reales)

Antes de la descarga completa, se mide el rendimiento con una muestra pequeña. Los archivos del benchmark se descargan directamente a su destino definitivo, de modo que contribuyen al progreso real.

Si la aceleración es suficiente (≈ ×4 o superior), se continúa con 8 workers para el resto. Si es baja, debe investigarse (posible límite de tasa de Drive) antes de lanzar la descarga completa.


In [7]:
cfg_local = LOCAL_DATASET / "nnUNetPlans_3d_fullres"
gt_local = LOCAL_DATASET / "gt_segmentations"
cfg_local.mkdir(parents=True, exist_ok=True)

N_BENCHMARK = 100
lote_secuencial = archivos_cfg[:N_BENCHMARK]
lote_paralelo = archivos_cfg[N_BENCHMARK:2 * N_BENCHMARK]

print("=== Benchmark: secuencial vs paralelo (8 workers) ===")

t0 = time.time()
descargar_lista(lote_secuencial, cfg_local)
t_secuencial = time.time() - t0
print(f"Secuencial:            {N_BENCHMARK} archivos en {t_secuencial/60:.2f} min "
      f"({t_secuencial/N_BENCHMARK:.2f} s/archivo)")

_, t_paralelo = descargar_lista_paralelo(lote_paralelo, cfg_local, n_workers=8, silencioso=True)
print(f"Paralelo (8 workers):  {N_BENCHMARK} archivos en {t_paralelo/60:.2f} min "
      f"({t_paralelo/N_BENCHMARK:.2f} s/archivo)")

aceleracion = t_secuencial / t_paralelo if t_paralelo > 0 else float("nan")
print(f"\nAceleracion observada: x{aceleracion:.1f}")

restantes = (len(archivos_cfg) - 2 * N_BENCHMARK) + len(archivos_gt)
estimado_min = restantes * (t_paralelo / N_BENCHMARK) / 60
print(f"Estimacion para los ~{restantes} archivos restantes con 8 workers: ~{estimado_min:.0f} min")

if aceleracion < 2:
    print("\nATENCION: la aceleracion es baja -- puede que Drive este limitando la tasa "
          "de peticiones. No subas el numero de workers todavia; revisa los fallos si los hay.")
elif aceleracion >= 4:
    print("\nBuena aceleracion. Se puede probar a subir a 12 workers en la siguiente celda "
          "si se quiere apurar mas, o seguir con 8 (mas conservador).")


=== Benchmark: secuencial vs paralelo (8 workers) ===


Secuencial:            100 archivos en 13.03 min (7.82 s/archivo)


Paralelo (8 workers):  100 archivos en 2.15 min (1.29 s/archivo)

Aceleracion observada: x6.1
Estimacion para los ~3800 archivos restantes con 8 workers: ~82 min

Buena aceleracion. Se puede probar a subir a 12 workers en la siguiente celda si se quiere apurar mas, o seguir con 8 (mas conservador).


## 8. Descarga del resto del dataset con la versión paralela

Se utilizan 8 workers (`N_WORKERS` puede elevarse a 12 si el benchmark anterior lo permite con seguridad). Todo lo descargado en el benchmark se omite automáticamente por tamaño.


In [8]:
N_WORKERS = 8

if LOCAL_DATASET.exists():
    print("Ya existe una copia local -- se reutiliza y se completa lo que falte (no se borra).")
LOCAL_DATASET.mkdir(parents=True, exist_ok=True)

pend_json, _ = descargar_lista_paralelo(archivos_json_top, LOCAL_DATASET, n_workers=N_WORKERS)
assert not pend_json, "Fallo al descargar los JSON de nivel superior."

t0 = time.time()
pend_cfg, _ = descargar_lista_paralelo(archivos_cfg, cfg_local, n_workers=N_WORKERS)
print(f"nnUNetPlans_3d_fullres: {(time.time()-t0)/60:.1f} min, {len(pend_cfg)} archivos sin descargar")
assert not pend_cfg, "Quedaron archivos sin descargar en nnUNetPlans_3d_fullres."

t0 = time.time()
pend_gt, _ = descargar_lista_paralelo(archivos_gt, gt_local, n_workers=N_WORKERS)
print(f"gt_segmentations: {(time.time()-t0)/60:.1f} min, {len(pend_gt)} archivos sin descargar")
assert not pend_gt, "Quedaron archivos sin descargar en gt_segmentations."

shutil.copy2(SPLIT_CORRECTO, LOCAL_DATASET / "splits_final.json")
print("\nCopia local lista (via API de Drive, en paralelo), incluido el split canonico (5 folds).")


Ya existe una copia local -- se reutiliza y se completa lo que falte (no se borra).
Total: 3 archivos, 8 workers en paralelo


  3/3 (ok=3 skip=0 fail=0)
Terminado en 0.04 min. ok=3 skip=0 fail=0
Total: 3000 archivos, 8 workers en paralelo
  100/3000 (ok=0 skip=100 fail=0)
  200/3000 (ok=0 skip=200 fail=0)


  300/3000 (ok=100 skip=200 fail=0)
  400/3000 (ok=200 skip=200 fail=0)
  500/3000 (ok=300 skip=200 fail=0)
  600/3000 (ok=400 skip=200 fail=0)
  700/3000 (ok=500 skip=200 fail=0)
  800/3000 (ok=600 skip=200 fail=0)
  900/3000 (ok=700 skip=200 fail=0)
  1000/3000 (ok=800 skip=200 fail=0)
  1100/3000 (ok=900 skip=200 fail=0)
  1200/3000 (ok=1000 skip=200 fail=0)
  1300/3000 (ok=1100 skip=200 fail=0)
  1400/3000 (ok=1200 skip=200 fail=0)
  1500/3000 (ok=1300 skip=200 fail=0)
  1600/3000 (ok=1400 skip=200 fail=0)
  1700/3000 (ok=1500 skip=200 fail=0)
  1800/3000 (ok=1600 skip=200 fail=0)
  1900/3000 (ok=1700 skip=200 fail=0)
  2000/3000 (ok=1800 skip=200 fail=0)
  2100/3000 (ok=1900 skip=200 fail=0)
  2200/3000 (ok=2000 skip=200 fail=0)
  2300/3000 (ok=2100 skip=200 fail=0)


  2400/3000 (ok=2200 skip=200 fail=0)
  2500/3000 (ok=2300 skip=200 fail=0)
  2600/3000 (ok=2400 skip=200 fail=0)
  2700/3000 (ok=2500 skip=200 fail=0)
  2800/3000 (ok=2600 skip=200 fail=0)
  2900/3000 (ok=2700 skip=200 fail=0)


  3000/3000 (ok=2800 skip=200 fail=0)
Terminado en 53.60 min. ok=2800 skip=200 fail=0
nnUNetPlans_3d_fullres: 53.6 min, 0 archivos sin descargar
Total: 1000 archivos, 8 workers en paralelo


  100/1000 (ok=100 skip=0 fail=0)
  200/1000 (ok=200 skip=0 fail=0)
  300/1000 (ok=300 skip=0 fail=0)
  400/1000 (ok=400 skip=0 fail=0)
  500/1000 (ok=500 skip=0 fail=0)
  600/1000 (ok=600 skip=0 fail=0)
  700/1000 (ok=700 skip=0 fail=0)
  800/1000 (ok=800 skip=0 fail=0)
  900/1000 (ok=900 skip=0 fail=0)
  1000/1000 (ok=1000 skip=0 fail=0)
Terminado en 29.41 min. ok=1000 skip=0 fail=0
gt_segmentations: 29.4 min, 0 archivos sin descargar

Copia local lista (via API de Drive, en paralelo), incluido el split canonico (5 folds).


## 9. Verificación de la copia local (1000/1000) y del split canónico


In [9]:
local_cfg = LOCAL_DATASET / "nnUNetPlans_3d_fullres"
local_gt = LOCAL_DATASET / "gt_segmentations"

imgs_local = [p for p in local_cfg.glob("ImageCAS_*.b2nd") if not p.name.endswith("_seg.b2nd")]
segs_local = list(local_cfg.glob("ImageCAS_*_seg.b2nd"))
pkls_local = list(local_cfg.glob("ImageCAS_*.pkl"))
gts_local = list(local_gt.glob("ImageCAS_*.nii.gz"))

print("Imagenes:", len(imgs_local), "Segmentaciones:", len(segs_local), "PKL:", len(pkls_local), "GT:", len(gts_local))
assert len(imgs_local) == 1000 and len(segs_local) == 1000 and len(pkls_local) == 1000 and len(gts_local) == 1000

with open(LOCAL_DATASET / "splits_final.json") as f:
    split_local = json.load(f)
with open(SPLIT_CORRECTO) as f:
    split_canonico = json.load(f)
assert split_local == split_canonico, "El split local NO coincide con el canonico."
assert len(split_local) >= 5, "El split no tiene al menos 5 folds."

for fold in FOLDS:
    info = split_local[fold]
    print(f"Fold {fold}: {len(info['train'])} train / {len(info['val'])} val")
    assert len(info["train"]) == 800 and len(info["val"]) == 200

print("\nCopia local integra y split canonico confirmado para folds", FOLDS)


Imagenes: 1000 Segmentaciones: 1000 PKL: 1000 GT: 1000
Fold 2: 800 train / 200 val
Fold 3: 800 train / 200 val
Fold 4: 800 train / 200 val

Copia local integra y split canonico confirmado para folds [2, 3, 4]


## 10. Configuración de nnU-Net: datos en el SSD local y resultados en Drive


In [10]:
os.environ["nnUNet_preprocessed"] = str(LOCAL_PREP_ROOT)
os.environ["nnUNet_results"] = str(NNUNET_RESULTS)
os.environ["nnUNet_extTrainer"] = str(CUSTOM_TRAINERS)
NNUNET_RAW = BASE / "nnUNet_raw"
os.environ["nnUNet_raw"] = str(NNUNET_RAW)

print("nnUNet_preprocessed =", os.environ["nnUNet_preprocessed"])
print("nnUNet_results      =", os.environ["nnUNet_results"])
print("nnUNet_extTrainer   =", os.environ["nnUNet_extTrainer"])


nnUNet_preprocessed = /content/nnUNet_preprocessed
nnUNet_results      = /content/drive/MyDrive/TFM_nnUNet/nnUNet_results
nnUNet_extTrainer   = /content/drive/MyDrive/TFM_nnUNet/custom_trainers


## 11. Trainer v1 (idempotente, válido para cualquier fold)


In [11]:
TRAINER_TEMPLATE = r'''
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from nnunetv2.training.loss.compound_losses import DC_and_CE_loss
from nnunetv2.training.loss.deep_supervision import DeepSupervisionWrapper
from nnunetv2.training.loss.dice import MemoryEfficientSoftDiceLoss

from nnunetv2.training.nnUNetTrainer.variants.training_length.nnUNetTrainer_Xepochs import (
    nnUNetTrainer_{epochs}epochs
)


def soft_erode_3d(x):
    p1 = -F.max_pool3d(-x, kernel_size=(3, 1, 1), stride=1, padding=(1, 0, 0))
    p2 = -F.max_pool3d(-x, kernel_size=(1, 3, 1), stride=1, padding=(0, 1, 0))
    p3 = -F.max_pool3d(-x, kernel_size=(1, 1, 3), stride=1, padding=(0, 0, 1))
    return torch.minimum(torch.minimum(p1, p2), p3)


def soft_dilate_3d(x):
    return F.max_pool3d(x, kernel_size=3, stride=1, padding=1)


def soft_open_3d(x):
    return soft_dilate_3d(soft_erode_3d(x))


def soft_skeletonize_3d(x, iterations=10):
    opened = soft_open_3d(x)
    skeleton = F.relu(x - opened)
    for _ in range(iterations):
        x = soft_erode_3d(x)
        opened = soft_open_3d(x)
        delta = F.relu(x - opened)
        skeleton = skeleton + F.relu(delta - skeleton * delta)
    return skeleton


class SoftClDiceLoss(nn.Module):
    def __init__(self, iterations=10, smooth=1e-5):
        super().__init__()
        self.iterations = iterations
        self.smooth = smooth

    def forward(self, prediction, target):
        prediction = prediction.float()
        target = target.float()
        skel_pred = soft_skeletonize_3d(prediction, iterations=self.iterations)
        skel_true = soft_skeletonize_3d(target, iterations=self.iterations)
        tprec = ((skel_pred * target).sum() + self.smooth) / (skel_pred.sum() + self.smooth)
        tsens = ((skel_true * prediction).sum() + self.smooth) / (skel_true.sum() + self.smooth)
        cldice = (2.0 * tprec * tsens) / (tprec + tsens + self.smooth)
        return 1.0 - cldice


class DiceCEPlusSoftClDice(nn.Module):
    def __init__(self, regional_loss, lambda_cldice=0.2, skeleton_iterations=10):
        super().__init__()
        self.regional_loss = regional_loss
        self.lambda_cldice = lambda_cldice
        self.soft_cldice = SoftClDiceLoss(iterations=skeleton_iterations, smooth=1e-5)

    def forward(self, logits, target):
        loss_regional = self.regional_loss(logits, target)

        probabilities = torch.softmax(logits.float(), dim=1)
        foreground_probability = probabilities[:, 1:2]

        if target.ndim == logits.ndim - 1:
            target = target.unsqueeze(1)
        foreground_target = (target == 1).float()

        topology_loss = self.soft_cldice(foreground_probability, foreground_target)

        return loss_regional + self.lambda_cldice * topology_loss


class {class_name}(nnUNetTrainer_{epochs}epochs):

    def __init__(self, plans, configuration, fold, dataset_json, device=torch.device("cuda")):
        super().__init__(plans, configuration, fold, dataset_json, device)
        self.weight_dice = {weight_dice}
        self.weight_ce = {weight_ce}
        self.lambda_cldice = {lambda_cldice}
        self.skeleton_iterations = {skeleton_iterations}

    def on_train_start(self):
        super().on_train_start()
        self.print_to_log_file(
            f"[v1 fold {{self.fold}}] weight_dice={{self.weight_dice}} "
            f"weight_ce={{self.weight_ce}} lambda_cldice={{self.lambda_cldice}} "
            f"skeleton_iterations={{self.skeleton_iterations}}"
        )

    def _build_loss(self):
        regional_loss = DC_and_CE_loss(
            soft_dice_kwargs={{
                "batch_dice": self.configuration_manager.batch_dice,
                "smooth": 1e-5,
                "do_bg": False,
                "ddp": self.is_ddp
            }},
            ce_kwargs={{}},
            weight_ce=self.weight_ce,
            weight_dice=self.weight_dice,
            ignore_label=self.label_manager.ignore_label,
            dice_class=MemoryEfficientSoftDiceLoss
        )

        loss = DiceCEPlusSoftClDice(
            regional_loss=regional_loss,
            lambda_cldice=self.lambda_cldice,
            skeleton_iterations=self.skeleton_iterations,
        )

        if self.enable_deep_supervision:
            deep_supervision_scales = self._get_deep_supervision_scales()
            weights = np.array([1 / (2 ** i) for i in range(len(deep_supervision_scales))])
            weights[-1] = 0
            weights = weights / weights.sum()
            loss = DeepSupervisionWrapper(loss, weights)

        return loss
'''

codigo = TRAINER_TEMPLATE.format(
    class_name=V1_TRAINER,
    epochs=100,
    weight_dice=WEIGHT_DICE,
    weight_ce=WEIGHT_CE,
    lambda_cldice=LAMBDA_CLDICE,
    skeleton_iterations=SKELETON_ITERATIONS,
)
ruta_trainer = CUSTOM_TRAINERS / f"{V1_TRAINER}.py"

if ruta_trainer.exists():
    codigo_previo = ruta_trainer.read_text()
    if codigo_previo == codigo:
        print(f"El trainer ya existe en Drive y es IDENTICO byte a byte al que generaria ahora: {ruta_trainer}")
        print("No se sobrescribe -- se usa el mismo archivo que ya entreno fold 0 y fold 1.")
    else:
        raise RuntimeError(
            f"ATENCION: {ruta_trainer} ya existe pero su contenido NO coincide con el trainer "
            "que se acaba de generar en esta celda. Si sigues adelante sin resolver esto, "
            "fold 2/3/4 se entrenarian con una perdida distinta a la de fold 0/1 y la "
            "validacion cruzada dejaria de ser valida. Revisa manualmente la diferencia "
            "(por ejemplo con difflib) antes de continuar; NO se ha modificado el archivo."
        )
else:
    ruta_trainer.write_text(codigo)
    print(f"Trainer nuevo escrito en: {ruta_trainer}")
    print("(No existia una version previa en Drive con la que comparar -- "
          "asegurate de que este codigo es el mismo que uso fold 0/1.)")


El trainer ya existe en Drive y es IDENTICO byte a byte al que generaria ahora: /content/drive/MyDrive/TFM_nnUNet/custom_trainers/nnUNetTrainerV1_1000casos_100epochs.py
No se sobrescribe -- se usa el mismo archivo que ya entreno fold 0 y fold 1.


## 12. Funciones de tolerancia a interrupciones: estado de cada fold y trainer en Drive

Se realizan dos comprobaciones, ambas mediante API (rápidas, solo metadatos, sin FUSE):

- **`ya_esta_entrenado`:** además de contar los `.nii.gz`, exige la existencia de `summary.json` en la carpeta de validación, que nnU-Net escribe únicamente cuando la validación finaliza correctamente (200 predicciones sin `summary.json` pueden indicar una interrupción durante su escritura).
- **`tiene_checkpoint`:** comprueba si en la carpeta del fold (no en la de validación) existe `checkpoint_final.pth` o `checkpoint_latest.pth`. Esto determina si un entrenamiento pendiente se lanza con `--c` (continuación desde ese checkpoint) o desde cero, ya que una carpeta puede existir vacía o incompleta sin que haya nada que continuar.


In [12]:
id_results = buscar_id("nnUNet_results", id_tfm)

def _id_carpeta_fold(trainer_name, fold):
    id_dataset_res = buscar_id(DATASET_FOLDER_NAME, id_results)
    id_trainer_cfg = buscar_id(f"{trainer_name}__nnUNetPlans__{CONFIGURATION}", id_dataset_res)
    return buscar_id(f"fold_{fold}", id_trainer_cfg)

def ya_esta_entrenado(trainer_name, fold):
    try:
        id_fold = _id_carpeta_fold(trainer_name, fold)
        id_val = buscar_id("validation", id_fold)
        archivos_val = listar_hijos(id_val)
        n_preds = len([f for f in archivos_val if f["name"].endswith(".nii.gz")])
        tiene_summary = any(f["name"] == "summary.json" for f in archivos_val)
        return n_preds == 200 and tiene_summary
    except FileNotFoundError:
        return False

def tiene_checkpoint(trainer_name, fold):
    try:
        id_fold = _id_carpeta_fold(trainer_name, fold)
        archivos_fold = listar_hijos(id_fold)
        return any(f["name"] in ("checkpoint_final.pth", "checkpoint_latest.pth") for f in archivos_fold)
    except FileNotFoundError:
        return False

for fold in FOLDS:
    for trainer in (BASELINE_TRAINER, V1_TRAINER):
        if ya_esta_entrenado(trainer, fold):
            estado = "YA COMPLETO (200 preds + summary.json) -- se saltara"
        elif tiene_checkpoint(trainer, fold):
            estado = "a medias -- tiene checkpoint, se continuara con --c"
        else:
            estado = "pendiente -- se entrenara desde cero"
        print(f"fold {fold} / {trainer}: {estado}")


fold 2 / nnUNetTrainer_100epochs: pendiente -- se entrenara desde cero
fold 2 / nnUNetTrainerV1_1000casos_100epochs: pendiente -- se entrenara desde cero
fold 3 / nnUNetTrainer_100epochs: pendiente -- se entrenara desde cero
fold 3 / nnUNetTrainerV1_1000casos_100epochs: pendiente -- se entrenara desde cero
fold 4 / nnUNetTrainer_100epochs: pendiente -- se entrenara desde cero
fold 4 / nnUNetTrainerV1_1000casos_100epochs: pendiente -- se entrenara desde cero


## 13. Comprobación final: parada de seguridad antes del entrenamiento

La celda solo imprime un resumen y **no entrena**. Antes de ejecutar la sección 14 (entrenamiento), se revisa la salida para confirmar que dataset, split, trainer v1 y lista de entrenamientos pendientes son los esperados. Si alguno no coincide, el proceso se detiene en este punto.


In [13]:
print("=" * 40)
print("COMPROBACION FINAL")
print("=" * 40)
print(f"\nDataset: {DATASET_FOLDER_NAME}")
print(f"Preprocesado local: {len(imgs_local)} imagenes, {len(segs_local)} segmentaciones, "
      f"{len(pkls_local)} pkl, {len(gts_local)} GT")

print("\nSplit:")
for fold in FOLDS:
    info = split_local[fold]
    print(f"  Fold {fold}: {len(info['train'])} train / {len(info['val'])} val")

print(f"\nTrainer baseline: {BASELINE_TRAINER} (built-in)")
print(f"Trainer v1: {V1_TRAINER}")
print(f"  lambda_cldice = {LAMBDA_CLDICE}")
print(f"  skeleton_iterations = {SKELETON_ITERATIONS}")
print(f"  weight_dice = {WEIGHT_DICE}, weight_ce = {WEIGHT_CE}")
print(f"  Archivo: {ruta_trainer} (verificado identico al de fold 0/1 en la Seccion 11)")

print("\nEstado de cada fold+trainer:")
pendientes = []
for fold in FOLDS:
    for trainer in (BASELINE_TRAINER, V1_TRAINER):
        if ya_esta_entrenado(trainer, fold):
            print(f"  fold {fold} / {trainer}: completo")
        else:
            cont = tiene_checkpoint(trainer, fold)
            pendientes.append((fold, trainer, cont))
            print(f"  fold {fold} / {trainer}: PENDIENTE ({'continuar' if cont else 'desde cero'})")

print("\nPendientes de entrenar en la Seccion 14:")
if pendientes:
    for fold, trainer, cont in pendientes:
        print(f"  - fold {fold} / {trainer} ({'--c' if cont else 'desde cero'})")
else:
    print("  (ninguno -- los 6 ya estan completos)")

print("\nLISTO PARA ENTRENAR. Si todo lo de arriba es correcto, continuar con la Seccion 14.")


COMPROBACION FINAL

Dataset: Dataset503_ImageCAS1000
Preprocesado local: 1000 imagenes, 1000 segmentaciones, 1000 pkl, 1000 GT

Split:
  Fold 2: 800 train / 200 val
  Fold 3: 800 train / 200 val
  Fold 4: 800 train / 200 val

Trainer baseline: nnUNetTrainer_100epochs (built-in)
Trainer v1: nnUNetTrainerV1_1000casos_100epochs
  lambda_cldice = 0.2
  skeleton_iterations = 10
  weight_dice = 1.0, weight_ce = 1.0
  Archivo: /content/drive/MyDrive/TFM_nnUNet/custom_trainers/nnUNetTrainerV1_1000casos_100epochs.py (verificado identico al de fold 0/1 en la Seccion 11)

Estado de cada fold+trainer:
  fold 2 / nnUNetTrainer_100epochs: PENDIENTE (desde cero)
  fold 2 / nnUNetTrainerV1_1000casos_100epochs: PENDIENTE (desde cero)
  fold 3 / nnUNetTrainer_100epochs: PENDIENTE (desde cero)
  fold 3 / nnUNetTrainerV1_1000casos_100epochs: PENDIENTE (desde cero)
  fold 4 / nnUNetTrainer_100epochs: PENDIENTE (desde cero)
  fold 4 / nnUNetTrainerV1_1000casos_100epochs: PENDIENTE (desde cero)

Pendientes d

## 14. Entrenamiento de los folds 2, 3 y 4 (baseline y v1), omitiendo lo ya completado

Cada entrenamiento se lanza con `subprocess.run(..., check=True)`: si `nnUNetv2_train` falla (código de salida distinto de 0), el bucle se detiene con un error explícito en lugar de continuar con el siguiente fold sobre una base errónea.

Si el runtime se interrumpe durante un entrenamiento, el notebook completo se vuelve a ejecutar desde la sección 2: la descarga (secciones 7-8) no repite lo ya descargado, y las secciones 11-12 recalculan automáticamente qué está completo, qué tiene checkpoint y qué falta.


In [14]:
for fold in FOLDS:
    for trainer in (BASELINE_TRAINER, V1_TRAINER):
        if ya_esta_entrenado(trainer, fold):
            print(f"\n=== fold {fold} / {trainer}: ya completo, se salta ===")
            continue

        continuar = tiene_checkpoint(trainer, fold)
        flag_continuar = " --c" if continuar else ""

        print(f"\n=== Entrenando fold {fold} / {trainer} (continuar={continuar}) ===")
        cmd = f"nnUNetv2_train {DATASET_ID} {CONFIGURATION} {fold} -tr {trainer}{flag_continuar}"
        print(">", cmd)
        subprocess.run(cmd, shell=True, check=True)
        print(f"=== fold {fold} / {trainer}: entrenamiento terminado ===")

print("\nTodos los folds solicitados (o ya completos de antes) estan listos.")



=== Entrenando fold 2 / nnUNetTrainer_100epochs (continuar=False) ===
> nnUNetv2_train 503 3d_fullres 2 -tr nnUNetTrainer_100epochs


=== fold 2 / nnUNetTrainer_100epochs: entrenamiento terminado ===



=== Entrenando fold 2 / nnUNetTrainerV1_1000casos_100epochs (continuar=False) ===
> nnUNetv2_train 503 3d_fullres 2 -tr nnUNetTrainerV1_1000casos_100epochs


=== fold 2 / nnUNetTrainerV1_1000casos_100epochs: entrenamiento terminado ===



=== Entrenando fold 3 / nnUNetTrainer_100epochs (continuar=False) ===
> nnUNetv2_train 503 3d_fullres 3 -tr nnUNetTrainer_100epochs


...


=== Entrenando fold 3 / nnUNetTrainerV1_1000casos_100epochs (continuar=False) ===
> nnUNetv2_train 503 3d_fullres 3 -tr nnUNetTrainerV1_1000casos_100epochs


=== fold 3 / nnUNetTrainerV1_1000casos_100epochs: entrenamiento terminado ===



=== Entrenando fold 4 / nnUNetTrainer_100epochs (continuar=False) ===
> nnUNetv2_train 503 3d_fullres 4 -tr nnUNetTrainer_100epochs


=== fold 4 / nnUNetTrainer_100epochs: entrenamiento terminado ===



=== Entrenando fold 4 / nnUNetTrainerV1_1000casos_100epochs (continuar=False) ===
> nnUNetv2_train 503 3d_fullres 4 -tr nnUNetTrainerV1_1000casos_100epochs
=== fold 4 / nnUNetTrainerV1_1000casos_100epochs: entrenamiento terminado ===

Todos los folds solicitados (o ya completos de antes) estan listos.


## Siguiente paso

Tras este notebook, los folds 0-4 (baseline y v1) están completos en Drive. El paso siguiente es su evaluación (notebook de evaluación parametrizado por fold) y la agregación de los 5 folds para el análisis de validación cruzada final (pool de 1000 casos y media y desviación estándar entre folds).


In [1]:
print("=== COMPROBACIÓN REAL EN DRIVE API ===")

id_results = buscar_id("nnUNet_results", id_tfm)
id_dataset = buscar_id("Dataset503_ImageCAS1000", id_results)

for trainer in (BASELINE_TRAINER, V1_TRAINER):
    print(f"\n{trainer}")

    id_trainer = buscar_id(
        f"{trainer}__nnUNetPlans__3d_fullres",
        id_dataset
    )

    for fold in [2, 3, 4]:
        try:
            id_fold = buscar_id(f"fold_{fold}", id_trainer)
            id_val = buscar_id("validation", id_fold)

            archivos = listar_hijos(id_val)

            preds = [
                f for f in archivos
                if f["name"].endswith(".nii.gz")
            ]

            summaries = [
                f for f in archivos
                if f["name"] == "summary.json"
            ]

            print(
                f"Fold {fold}: "
                f"{len(preds)} predicciones | "
                f"summary.json={'SI' if summaries else 'NO'}"
            )

        except Exception as e:
            print(f"Fold {fold}: ERROR -> {e}")

=== COMPROBACIÓN REAL EN DRIVE API ===


NameError: name 'buscar_id' is not defined